In [7]:
# IMPORT STATEMENTS
import cv2
import re
import numpy as np
import matplotlib
from brokenaxes import brokenaxes 
matplotlib.use('pdf')
from matplotlib import pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable, axes_size
from mpl_toolkits.axes_grid1 import SubplotDivider, Size
from mpl_toolkits.axes_grid1.mpl_axes import Axes
import matplotlib.patches as patches
import matplotlib.colors as colors
from matplotlib.transforms import *
import PIL
import math
#get_ipython().magic(u'matplotlib inline')
import pandas as pd
import seaborn as sns
import json
from sklearn.metrics import *
from scipy.stats import *
from pprint import pprint
import os
import pickle
import sys
#sys.path.append("/booleanfs/sahoo/scripts/")
sys.path.append("/home/saptarshi.sinha/Hegemon/")
import StepMiner as smn
import HegemonUtil as hu
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

try:
    reload  # Python 2.7
except NameError:
    try:
        from importlib import reload  # Python 3.4+
    except ImportError:
        from imp import reload  # Python 3.0 - 3.3

def getPDF(cfile):
    import bone
    reload(bone)
    from matplotlib.backends.backend_pdf import PdfPages

    pdf = PdfPages(cfile)
    return pdf

def closePDF(pdf):
    import datetime
    d = pdf.infodict()
    d['Title'] = 'Plots'
    d['Author'] = 'Daniella Vo'
    d['Subject'] = "Microbe Polyp"
    d['Keywords'] = 'disease training validation ROC'
    d['CreationDate'] = datetime.datetime(2021, 1, 18)
    d['ModDate'] = datetime.datetime.today()
    pdf.close()


In [9]:
import bone
reload(bone)
class MacAnalysis(bone.MacAnalysis):

    def __init__(self):
        bone.MacAnalysis.__init__(self)
   

    def getNoble2010(self, tn=1):
        self.prepareData("PLP58")
        atype = self.h.getSurvName("c src1")
        atypes = ['A', 'D', 'S', 'T', 'A_CD', 'D_CD', 'S_CD', 'T_CD']
        ahash = {'ascending colon biopsy from healthy subject':0,
                'descending colon biopsy from healthy subject':1,
                'sigmoid colon biopsy from healthy subject':2,
                'terminal ileum biopsy from healthy subject':3,
                'ascending colon biopsy from crohns disease subject':4,
                'descending colon biopsy from crohns disease subject':5,
                'sigmoid colon biopsy from crohns disease subject':6,
                'terminal ileum biopsy from crohns disease subject':7}
        if (tn == 2):
            atypes = ['A', 'D']
            ahash = {'ascending colon biopsy from healthy subject':0,
                    'descending colon biopsy from healthy subject':1}
        self.initData(atype, atypes, ahash)     

    def getYang2020(self, tn=1):
        self.prepareData("COV279")
        atype = self.h.getSurvName("c diagnosis")
        atypes = ['HC', 'IPF', 'CHP']
        ahash = {'chp':2, 'ipf':1, 'control':0}
        aval = [ahash[i] if i in ahash else None for i in atype]
        if (tn == 2):
            atype = self.h.getSurvName("c Sex")
            atypes = ['F', 'M']
            ahash = {'male':1, 'female':0}
        if (tn == 3):
            atype = self.h.getSurvName('c race (hispanic;1, black;3,asian;4, white;5, other;6)')
            atypes = ['W', 'B', 'A', 'O']
            ahash = {'5':0, '1':3, '3':1, '4':2, 'UNKNOWN':3, '6':3}
        self.initData(atype, atypes, ahash)        
        
        
    def getYao2020IPF(self, tn=1, tb=1):
        self.prepareData('COV268')
        atype = self.h.getSurvName("c group")
        atypes = ['control', 'IPF']
        ahash = {}
        self.initData(atype, atypes, ahash)
        
    def getYao2020bulk(self, tn=1, tb=1):
        self.prepareData('COV268')
        atype = self.h.getSurvName("c Cell Type")
        #ahash = {'IgG CTC chip': 0}
        ahash = {'bulk':0}
        hval = [1 if i in ahash else None for i in atype]         
        atype = self.h.getSurvName("c group")
        atypes = ['control', 'IPF']
        ahash = {}
        atype = [atype[i] if hval[i] == 1
                else None for i in range(len(atype))]   
        self.initData(atype, atypes, ahash)     
        
        
    def getXu2020CoV2(self, tn=1, tb=0):
        self.prepareData("COV339")
        atype = self.h.getSurvName("c disease condition")
        atypes = ['H', 'CoV', 'Hp', 'IPF', 'Ma', 'Ssa']
        ahash = {'Hypersensitivity pneumonitis':2,
                'Donor':0,
                'Idiopathic pulmonary fibrosis':3,
                'Myositis-associated interstitial lng disease':4,
                'Systemic slcerosis-associated interstitial lung disease':5, '':1}
        if (tn == 2):
            atypes = ['H', 'CoV', 'IPF']
            ahash = {'Donor':0, 'Idiopathic pulmonary fibrosis':2, '':1}
        if (tn == 3):
            atypes = ['H', 'CoV']
            ahash = {'Donor':0, '':1}
        if (tn == 4):
            atypes = ['H', 'IPF']
            ahash = {'Donor':0, 'Idiopathic pulmonary fibrosis':1}
        if (tn == 5):
            atypes = ['IPF', 'CoV']
            ahash = {'Idiopathic pulmonary fibrosis':0, '':1}
        self.initData(atype, atypes, ahash)        
        
    
def plotViolinBar(ana, desc=None):
    fig = plt.figure(figsize=(5,5), dpi=100)
    plt.subplots_adjust(hspace=0.5, wspace=0.5)
    ax1 = plt.subplot2grid((4, 1), (0, 0))
    ax2 = plt.subplot2grid((4, 1), (1, 0), rowspan=3)
    params = {'spaceAnn': len(ana.order)/len(ana.atypes), 'tAnn': 1, 'widthAnn':1,
              'genes': [], 'ax': ax1, 'acolor': acolor}
    ax = ana.printTitleBar(params)
    res = ana.getROCAUC()
    ax.text(len(ana.cval[0]), 4, res)
    if desc is not None:
        ax.text(-1, 2, desc, horizontalalignment='right',
                    verticalalignment='center')
    params = {'spaceAnn': len(ana.order)/len(ana.atypes), 'tAnn': 1, 'widthAnn':1,
            'genes': [], 'ax': ax2, 'acolor': acolor, 'vert': 0}
    ax = ana.printViolin(None, params)
    return fig

def plotDensityBar(ana, desc=None):
    fig = plt.figure(figsize=(4,4), dpi=100)
    plt.subplots_adjust(hspace=0.5, wspace=0.5)
    ax1 = plt.subplot2grid((4, 1), (0, 0))
    ax2 = plt.subplot2grid((4, 1), (1, 0), rowspan=3)
    params = {'spaceAnn': len(ana.order)/len(ana.atypes), 'tAnn': 1, 'widthAnn':1,
              'genes': [], 'ax': ax1, 'acolor': acolor}
    ax = ana.printTitleBar(params)
    res = ana.getMetrics(ana.cval[0])
    ax.text(len(ana.cval[0]), 4, ",".join(res))
    if desc is not None:
        ax.text(-1, 2, desc, horizontalalignment='right',
                    verticalalignment='center')
    ax = ana.densityPlot(ax2, acolor)
    return fig

def processData(ana, l1, wt1, desc=None, violin=1):
    ana.orderData(l1, wt1)
    if (violin == 1):
        return plotViolinBar(ana, desc)
    return plotDensityBar(ana, desc)

def processDataDf(ana, l1, wt1, desc=None):
    fig = plt.figure(figsize=(4,4), dpi=100)
    plt.subplots_adjust(hspace=0.5, wspace=0.5)
    ax1 = plt.subplot2grid((4, 1), (0, 0))
    ax2 = plt.subplot2grid((4, 1), (1, 0), rowspan=3)

    c_dict, fpr, tpr, roc_auc = bone.processGeneGroupsDf(ana, l1, wt1)
    params = {'spaceAnn': len(ana.order)/len(ana.atypes), 'tAnn': 1, 'widthAnn':1,
              'genes': [], 'ax': ax1, 'acolor': acolor}
    ax = ana.printTitleBar(params)
    res = ana.getROCAUC()
    ax.text(len(ana.cval[0]), 4, res)
    if desc is not None:
        ax.text(-1, 2, desc, horizontalalignment='right',
                    verticalalignment='center')
    params = {'spaceAnn': len(ana.order)/len(ana.atypes), 'tAnn': 1, 'widthAnn':1,
            'genes': [], 'ax': ax2, 'acolor': acolor, 'vert': 0}
    ax = ana.printViolin(None, params)
    return fig


def getOrder(ana, l1):
    from scipy.stats import fisher_exact, ttest_ind
    res = []
    for s in l1:
        for gn in s:
            id1 = ana.h.getBestID(ana.h.getIDs(gn).keys())
            if id1 is None:
                continue
            e = ana.h.getExprData(id1)
            v1 = np.array([float(e[i]) if e[i] != "" else 0 for i in ana.state[0]])
            v2 = np.array([float(e[i]) if e[i] != "" else 0 for i in ana.state[1]])
            t, p = ttest_ind(v1,v2,equal_var=False)
            res += [[id1, ana.h.getSimpleName(id1),
                   t, p, np.mean(v1)-np.mean(v2)]]
    return pd.DataFrame(res, columns=['ProbeID', 'Name', 'T', 'p', 'Diff'])
def plotVolcano(ana, genes, cfile, ylim=[0, 10.5], xlim=[-6, 6]):
    df = processGenes(ana.h, [ana.state[0], ana.state[1]], genes)
    df['Size'] = 10
    fig,ax = plt.subplots(figsize=(6,4), dpi=100)
    crcdf = pd.read_csv(cfile)
    crcdf['logp'] = -np.log10(crcdf['pvalue'])
    ax = sns.scatterplot('log2FoldChange', 'logp', size=0.1, color='0.8',
                         edgecolor="none", data=crcdf)
    ax.set_ylim(ylim)
    ax.set_xlim(xlim)
    ax.legend().set_visible(False)
    import io
    import base64
    buffer = io.BytesIO()
    fig.savefig(buffer, format='jpg')
    buffer.seek(0)
    volcano = base64.b64encode(buffer.read())
    from PIL import Image, ImageDraw
    buffer.seek(0)
    img = Image.open(buffer)
    x = list(ax.bbox.bounds)
    x[2] = x[2] + x[0]
    x[3] = x[3] + x[1] - 2
    x[1] = x[1] - 2
    img = img.crop(x)

    fig,ax = plt.subplots(figsize=(6,4), dpi=100)
    ax = sns.scatterplot('Diff', 'logp', hue='Diff', palette='vlag',
                         data=df, size='Size', size_norm=(0, 10), 
                         edgecolor="none", zorder=2, ax=ax)
    ax.legend().set_visible(False)
    ax.set_ylim(ylim)
    ax.set_xlim(xlim)
    for i in df.index:
        ax.text(df['Diff'][i]+.02, df['logp'][i], str(df['Name'][i]))
    ax.imshow(img,
              aspect = ax.get_aspect(),
              extent = ax.get_xlim() + ax.get_ylim(),
              zorder = 1) #put the map under the heatmap
    return ax

def savePList(ofile, ana, l1):
    df = getOrder(ana, l1)
    df1 = df.sort_values(by=['T'], ascending=True)
    bone.saveList(ofile, df1['Name'])

def getSViP():
    l1 = [bone.readList("/booleanfs2/sahoo/Data/Macrophage/BN/covid/iav-list-1.txt")[0:20]] # 20 gene signature
    wt1 = [1]
    return wt1, l1

def getViP():
    l1 = [bone.readList("/booleanfs2/sahoo/Data/Macrophage/BN/covid/list-2.txt")] # 166 gene signature
    wt1 = [1]
    return wt1, l1

import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
def getPDF(cfile):
    import bone
    reload(bone)
    from matplotlib.backends.backend_pdf import PdfPages

    pdf = PdfPages(cfile)
    return pdf

def closePDF(pdf):
    import datetime
    d = pdf.infodict()
    d['Title'] = 'Plots'
    d['Author'] = 'Daniella Vo'
    d['Subject'] = "Microbe Polyp"
    d['Keywords'] = 'disease training validation ROC'
    d['CreationDate'] = datetime.datetime(2021, 1, 18)
    d['ModDate'] = datetime.datetime.today()
    pdf.close()

In [10]:
import bone
reload(bone)
ana = MacAnalysis()
#ana.getYang2020()#BULK
ana.getGSE193677()


#ana.order = [i for i in ana.h.aRange()]
df = pd.DataFrame()

#ana.order = [i for i in ana.h.aRange()]
df = pd.DataFrame()
wt1, l1 =[1], [['CXCL1', 'CHI3L1', 'PDPN', 'CYGB', 'SERPINF1', 'CPZ', 'CCN4', 'LRRN2', 'CTSH', 'CXCL3', 'FMNL1', 'FAM20C', 'SEMA4D', 'CCDC69', 'IFITM1']]
ana.order = [i for i in ana.h.aRange()]
ana.orderData(l1, wt1)
thr = hu.getThrData(ana.f_ranks)
fthr = thr[0]
print (fthr)
df['C1'] = [None, None] + list(ana.f_ranks)
df['C1'] = np.array(df['C1']).astype(float)
df['FOR'] = np.where(df['C1']> fthr, 1, 0)# based on SM thesold

atype = ana.h.getSurvName('c demographics_gender_ch1')
ahash = {'Male':1, 'Female':0}
df['gender'] = [ahash[k] if k in ahash else None for k in atype]

atype = ana.h.getSurvName('c ibd_disease_ch1')
ahash = {'Control':0, 'CD_Pouch':2,'CD':1,'UC':1,'UC_Pouch':2}
df['disease'] = [ahash[k] if k in ahash else None for k in atype]

atype = ana.h.getSurvName('c ibd_endoseverity_4levels_ch1')
ahash = {'NA':0, 'Inactive':0, 'Mild':1, 'Moderate':2, 'Severe':3}
df['endoseverity'] = [ahash[k] if k in ahash else None for k in atype]

atype = ana.h.getSurvName('c ibd_clinicianmeasure_inactive_active_ch1')
ahash = {'NA':0, 'Active':2, 'Inactive':1}
df['inactive_active'] = [ahash[k] if k in ahash else None for k in atype]

atype = ana.h.getSurvName('c regionre_ch1')
ahash = {'Cecum':0, 'Ileum':1, 'LeftColon':0,'Rectum':0,'RightColon':0}
df['location'] = [ahash[k] if k in ahash else None for k in atype]

atype = ana.h.getSurvName('c diseasetypere_ch1')
ahash = {'CD.I':2, 'CD.NonI':1, 'Control.NonI':0,'UC.I':2,'UC.NonI':1}
df['diseasetypere_NI_I'] = [ahash[k] if k in ahash else None for k in atype]


df = df.drop(df[df.disease == 0].index)
df = df.drop(labels=[0,1], axis=0) #to delete duplicates
#atype = ana.h.getSurvName('c joint problems')
#ahash = {'FALSE':0, 'TRUE':1}
#df['joint_problems'] = [ahash[k] if k in ahash else None for k in atype]


#wt1, l1 = [1], [bone.getEntries('ipf/IPF signature_Bayesian 153 gene DOWN ONLY_PMID_21974901.txt', 0)]
#ana.orderData(l1, wt1)
#df['c1'] = [None, None] + list(ana.f_ranks)
#df['PMID_21974901dn'] = np.array(df['c1']).astype(float)





#df = df.drop(lasbels=[0,1], axis=0)

#df = df.drop(df[df['disease'] == '1'].index)
#df = df.drop(df[df.disease == 1].index)
#df = df.drop(labels=[0,1], axis=0) #to delete duplicates
#df['delta_VC']=df['followupVC'] - df['diagnosisVC']

#df1 = bone.printOLS("C1 ~ anatomic_location + inflammation_status + response + ethnicity + joint_problems + family_history + cdai + bradshaw + esr + crp + affected_relatives+ stoma_ileal + diagnosis_oral + diagnosis_ileal + diagnosis_colonic + diagnosis_rectal + diagnosis_anal_perianal + followup_oral +followup_ileal + followup_colonic + followup_rectal + followup_anal_perianal + delta_VC + behavior_at_diagnosis + disease_group + smoking_status", df)
df1 = bone.printOLS("FOR ~ diseasetypere_NI_I + location + endoseverity + inactive_active", df)
#df

GSE193677 (n = 2490)
GSE193677 http://hegemon.ucsd.edu/Tools/explore.php?key=blood:leukemia&id=SS65.5
947 461 374 112 SS65.5
[14]
-2.7738765287052773
                            OLS Regression Results                            
Dep. Variable:                    FOR   R-squared:                       0.126
Model:                            OLS   Adj. R-squared:                  0.124
Method:                 Least Squares   F-statistic:                     64.20
Date:                Fri, 17 Apr 2026   Prob (F-statistic):           9.32e-51
Time:                        13:00:45   Log-Likelihood:                -1063.2
No. Observations:                1783   AIC:                             2136.
Df Residuals:                    1778   BIC:                             2164.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|

In [12]:
import bone
reload(bone)
ana = MacAnalysis()
#ana.getYang2020()#BULK
ana.getGSE193677()


#ana.order = [i for i in ana.h.aRange()]
df = pd.DataFrame()

#ana.order = [i for i in ana.h.aRange()]
df = pd.DataFrame()
wt1, l1 =[1], [['CXCL1', 'CHI3L1', 'PDPN', 'CYGB', 'SERPINF1', 'CPZ', 'CCN4', 'LRRN2', 'CTSH', 'CXCL3', 'FMNL1', 'FAM20C', 'SEMA4D', 'CCDC69', 'IFITM1']]
ana.order = [i for i in ana.h.aRange()]
ana.orderData(l1, wt1)
thr = hu.getThrData(ana.f_ranks)
fthr = thr[0]
print (fthr)
df['C1'] = [None, None] + list(ana.f_ranks)
df['C1'] = np.array(df['C1']).astype(float)
df['FOR'] = np.where(df['C1']> fthr, 1, 0)# based on SM thesold

atype = ana.h.getSurvName('c demographics_gender_ch1')
ahash = {'Male':1, 'Female':0}
df['gender'] = [ahash[k] if k in ahash else None for k in atype]

atype = ana.h.getSurvName('c ibd_disease_ch1')
ahash = {'Control':0, 'CD_Pouch':2,'CD':1,'UC':1,'UC_Pouch':2}
df['disease'] = [ahash[k] if k in ahash else None for k in atype]

atype = ana.h.getSurvName('c ibd_endoseverity_4levels_ch1')
ahash = {'NA':0, 'Inactive':0, 'Mild':1, 'Moderate':2, 'Severe':3}
df['endoseverity'] = [ahash[k] if k in ahash else None for k in atype]

atype = ana.h.getSurvName('c ibd_clinicianmeasure_inactive_active_ch1')
ahash = {'NA':0, 'Active':2, 'Inactive':1}
df['inactive_active'] = [ahash[k] if k in ahash else None for k in atype]

atype = ana.h.getSurvName('c regionre_ch1')
ahash = {'Cecum':0, 'Ileum':1, 'LeftColon':0,'Rectum':0,'RightColon':0}
df['location'] = [ahash[k] if k in ahash else None for k in atype]

atype = ana.h.getSurvName('c diseasetypere_ch1')
ahash = {'CD.I':2, 'CD.NonI':1, 'Control.NonI':0,'UC.I':2,'UC.NonI':1}
df['diseasetypere_NI_I'] = [ahash[k] if k in ahash else None for k in atype]


df = df.drop(df[df.disease == 0].index)
df = df.drop(labels=[0,1], axis=0) #to delete duplicates


Udf1 = bone.printOLS("FOR ~ gender", df)
Udf2 = bone.printOLS("FOR ~ disease", df)
Udf3 = bone.printOLS("FOR ~ endoseverity", df)
Udf4 = bone.printOLS("FOR ~ inactive_active", df)
Udf5 = bone.printOLS("FOR ~ location", df)
Udf6 = bone.printOLS("FOR ~ diseasetypere_NI_I", df)



UDF=pd.concat([Udf1, Udf2, Udf3, Udf4, Udf6], axis=0)

GSE193677 (n = 2490)
GSE193677 http://hegemon.ucsd.edu/Tools/explore.php?key=blood:leukemia&id=SS65.5
947 461 374 112 SS65.5
[14]
-2.7738765287052773
                            OLS Regression Results                            
Dep. Variable:                    FOR   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.2820
Date:                Fri, 17 Apr 2026   Prob (F-statistic):              0.595
Time:                        13:01:22   Log-Likelihood:                -1352.0
No. Observations:                2029   AIC:                             2708.
Df Residuals:                    2027   BIC:                             2719.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      

In [14]:
import bone
import pandas as pd
import matplotlib.pyplot as plt
cfile = "MF_15gene_test.txt"
df = pd.read_table(cfile)
#labels, index = np.unique(df["Number"], return_inverse=True)
#df['Bx Source'] = np.array(df['Bx_Source']).astype(float)



#df1 = bone.printOLS("Group ~ Bx_Source + Diagonosis + CD_Behavior + CD_location + Rx_Response + Treatment_Type + PDO_Subtype + SES_Initial + SES_Follow + SES_Delta + Stricture + Adv_Therapy", df)
#df1 = bone.printOLS("SiglecXII ~ Position", df)
#df[['Group','SES_Delta']]

Udf1 = bone.printOLS("CS15gene ~ Sex", df)
Udf2 = bone.printOLS("CS15gene ~ Diagonosis", df)
Udf3 = bone.printOLS("CS15gene ~ Ethnicity", df)
Udf4 = bone.printOLS("CS15gene ~ Age_at_Diversion", df)
Udf7 = bone.printOLS("CS15gene ~ Treatment_Type", df)
Udf8 = bone.printOLS("CS15gene ~ Anti_TNF_Rx_Response", df)
Udf9 = bone.printOLS("CS15gene ~ Adv_Therapy", df)
#df10 = bone.printOLS("C1 ~ Treatment_Type", df)
#df11 = bone.printOLS("C1 ~ Anti_TNF_Rx_Response", df)
#Udf12 = bone.printOLS("C1 ~ Adv_Therapy", df)
Udf13 = bone.printOLS("CS15gene ~ Stricture", df)
#Udf14 = bone.printOLS("C1 ~ NInitial_SES_UCEIS", df)
#Udf15 = bone.printOLS("C1 ~ NFollow_SES_USCIS", df)
#Udf16 = bone.printOLS("C1 ~ NDelta_SES_UCEIS", df)
Udf17 = bone.printOLS("CS15gene ~ Ninitial_SES_Mayo_Sub", df)
Udf18 = bone.printOLS("CS15gene ~ NFollow_SES_Mayo_Sub", df)
#Udf19 = bone.printOLS("C1 ~ NDelta_SES_Mayo_Sub", df)
Udf20 = bone.printOLS("CS15gene ~ NInitial_PRO_Partial_Mayo", df)
Udf21=bone.printOLS("CS15gene ~ NFollow_PRO_Partial_Mayo", df)
#Udf22= bone.printOLS("C1 ~ NDelta_PRO_Partial_Mayo", df)
#df22 = bone.printOLS("C1 ~ SES_Ileum_Delta", df)
#Udf23 = bone.printOLS("C1 ~ PRO_Initial", df)
#Udf24 = bone.printOLS("C1 ~ PRO_Follow", df)
#df25 = bone.printOLS("C1 ~ PRO_Delta", df)

UDF=pd.concat([Udf1, Udf2, Udf3, Udf4, Udf7, Udf8, Udf9, Udf13, Udf17, Udf18, Udf20, Udf21], axis=0)
#UDF

                            OLS Regression Results                            
Dep. Variable:               CS15gene   R-squared:                       0.068
Model:                            OLS   Adj. R-squared:                  0.058
Method:                 Least Squares   F-statistic:                     7.127
Date:                Fri, 17 Apr 2026   Prob (F-statistic):            0.00889
Time:                        13:06:26   Log-Likelihood:                -283.65
No. Observations:                 100   AIC:                             571.3
Df Residuals:                      98   BIC:                             576.5
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -3.8518      0.988     -3.899      0.0

In [15]:
import bone
import pandas as pd
import matplotlib.pyplot as plt
cfile = "MF_15gene_test.txt"
df = pd.read_table(cfile)
#labels, index = np.unique(df["Number"], return_inverse=True)
#df['Bx Source'] = np.array(df['Bx_Source']).astype(float)
#l3, i3 = np.unique(df["Diagonosis"], return_inverse=True)
#df["Diagonosis"] = i3
#l2, i2 = np.unique(df["Bx Source"], return_inverse=True)
#df["Bx_Source"] = i2 
#l4, i4 = np.unique(df["UC Extent"], return_inverse=True)
#df["UC_Extent"] = i4 
#l5, i5 = np.unique(df["Rx Response"], return_inverse=True)
#df["Rx_Response"] = i5 
#l6, i6 = np.unique(df["Rx Response"], return_inverse=True)
#df["Rx_Response"] = i6 
#l7, i7 = np.unique(df["Group"], return_inverse=True)
#df["Group"] = i7
#l8, i8 = np.unique(df["Treatment Type"], return_inverse=True)
#df["Treatment_Type"] = i8




df1 = bone.printOLS("CS15gene ~ Adv_Therapy + Ninitial_SES_Mayo_Sub + NFollow_SES_Mayo_Sub + NFollow_PRO_Partial_Mayo", df)
#df1 = bone.printOLS("SiglecXII ~ Position", df)
#df[['Group','SES_Delta']]

                            OLS Regression Results                            
Dep. Variable:               CS15gene   R-squared:                       0.301
Model:                            OLS   Adj. R-squared:                  0.272
Method:                 Least Squares   F-statistic:                     10.23
Date:                Fri, 17 Apr 2026   Prob (F-statistic):           6.21e-07
Time:                        13:06:50   Log-Likelihood:                -269.24
No. Observations:                 100   AIC:                             548.5
Df Residuals:                      95   BIC:                             561.5
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               